# Week 10 – Baseline Backtest

Backtests two baseline strategies on every ticker in `TICKERS`:
1. Buy-and-hold
2. SMA crossover (50-day vs 200-day)

Each ticker/strategy is a separate single-asset run starting with $10,000. Metrics come from `src/evaluation/metrics.py`.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Project root = nearest parent folder that contains src/ (works from any cwd).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

# ---- Tickers to analyse (edit this list) -----------------------------------
TICKERS = ["AAPL", "MSFT", "GOOGL"]

from zipline import run_algorithm
from src.evaluation.csvdir_prep import prepare_csvdir
from src.evaluation.zipline_baselines import (
    ensure_bundle, make_buy_and_hold, make_sma_crossover, trade_pnls,
    utc, show_error,
)
from src.evaluation.metrics import compute_sharpe, compute_max_drawdown, compute_win_rate
# ---- Data -------------------------------------------------------------------
# A 50/200-day SMA needs years of history, but data/processed/indicators only
# holds ~60 trading days per ticker. So long Yahoo history is downloaded once
# into <root>/data/csvdir/daily/ (cached afterwards).
# Set SOURCE = "indicators" to convert your indicator CSVs instead (short windows only).
DATA_START, DATA_END = "2014-01-01", "2024-12-31"
SOURCE = "yahoo"                       # or "indicators"
CSV_DIR = str(ROOT / "data" / "csvdir")
INDICATORS_DIR = str(ROOT / "data" / "processed" / "indicators")

prepare_csvdir(TICKERS, CSV_DIR, source=SOURCE, start=DATA_START, end=DATA_END,
               indicators_dir=INDICATORS_DIR)
BUNDLE = ensure_bundle(CSV_DIR)        # re-ingests automatically if the tickers change

In [ ]:
start = utc("2015-01-01")
end   = utc("2024-12-30")   # last bar in the Yahoo download
CAPITAL = 10_000

STRATEGIES = {
    "Buy-and-hold": lambda t: make_buy_and_hold(t),
    "SMA 50/200":   lambda t: make_sma_crossover(t, fast=50, slow=200),
}

def run(strategy):
    initialize, handle_data = strategy
    return run_algorithm(start=start, end=end, initialize=initialize,
                         handle_data=handle_data, capital_base=CAPITAL, bundle=BUNDLE)

results = {}          # (ticker, strategy) -> zipline result
for t in TICKERS:
    for name, factory in STRATEGIES.items():
        try:
            results[(t, name)] = run(factory(t))
        except Exception as exc:
            show_error(f"{t} / {name}", exc)

In [ ]:
rows = []
for (t, name), res in results.items():
    equity, returns = res.portfolio_value, res.returns
    pnls = trade_pnls(res)          # closed round trips only
    rows.append({
        "Ticker": t, "Strategy": name,
        "Sharpe": f"{compute_sharpe(returns):.3f}",
        "Max Drawdown": f"{compute_max_drawdown(equity):.2%}",
        "Win Rate": f"{compute_win_rate(pnls):.2%}" if pnls else "N/A",
        "Trades": len(pnls),
        "Total Return": f"{equity.iloc[-1] / CAPITAL - 1:.2%}",
    })
summary = pd.DataFrame(rows).set_index(["Ticker", "Strategy"])
display(summary)

In [ ]:
n = len({t for t, _ in results})
fig, axes = plt.subplots(max(n, 1), 1, figsize=(12, 4 * max(n, 1)), squeeze=False)
for ax, t in zip(axes[:, 0], [t for t in TICKERS if any(k[0] == t for k in results)]):
    for name in STRATEGIES:
        if (t, name) in results:
            eq = results[(t, name)].portfolio_value
            ax.plot(eq.index, eq.to_numpy(dtype=float), label=name)
    ax.set_title(f"Equity Curves - {t}")
    ax.set_ylabel("Portfolio Value ($)")
    ax.legend(); ax.grid(True)
axes[-1, 0].set_xlabel("Date")
plt.tight_layout()
plt.show()